In [ ]:
# Import Data and Setup
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

file_path = "/content/drive/MyDrive/clash_royale_reviews.csv"

df = pd.read_csv(
    file_path,
    usecols=["review_id", "content", "score", "at", "likes", "appVersion"],
    parse_dates=["at"]
)

In [ ]:
# Categorize Reviews Based On Score
conditions = [
(df['score'] > 2),
(df['score'] < 3)
]
choices = [
'positive',
'negative'
]
df['review_category'] = np.select(conditions, choices, default = '')


In [ ]:
#sample smaller portions of reviews to avoid crashing during analysis
sampledf = df.sample(n= 500000, random_state = 123)
smaller_sampledf = df.sample(n = 100000, random_state = 123)
print(smaller_sampledf)
vectorizer = CountVectorizer(min_df=0.005, max_df=0.9)
clash_dtm = vectorizer.fit_transform(sampledf['content'].values.astype('U'))

                                    review_id  \
1954910  78270015-b6b2-44e7-bf04-fc5bcb3c3282   
1693941  a4da1d66-9a29-4b19-bd0d-5313e0d94680   
1015639  f0bed8fe-d7bf-43ef-9508-ba3b7fcca225   
376978   782836c0-e4e7-4219-873c-2eafb224a5cc   
1730292  2963e7b7-0714-4f08-89f1-4b50c2385aae   
...                                       ...   
1324887  562f54f6-d204-46fc-95a0-1af94bda0aaa   
1914958  1be4531c-e812-4ce7-ba21-68a67b9e61b5   
636033   89fea8b4-6a7a-4e3d-824c-2b4c19e4d5c9   
1606659  8cf53035-ab22-4332-a175-b7ebbdb17327   
672727   178efdb8-35da-4da8-bbbd-6435dfd55b4e   

                                                   content  score  \
1954910                                 Love this game!!!!      5   
1693941  It's like if Clash of Clans and Hearthstone ha...      4   
1015639                                I like Clash Royale      5   
376978                                    I love this game      5   
1730292  It's quick fun addicting and well thought out ...      5 

In [ ]:
import nltk
nltk.download('opinion_lexicon')

[nltk_data] Downloading package opinion_lexicon to /root/nltk_data...
[nltk_data]   Unzipping corpora/opinion_lexicon.zip.


True

In [ ]:
from nltk.corpus import opinion_lexicon
pos_list=set(opinion_lexicon.positive())
neg_list=set(opinion_lexicon.negative())

In [ ]:
# setup pandas dataframe to use lexicon on
vocab = vectorizer.get_feature_names_out().tolist()
lexicon_df_dtm = pd.DataFrame(data = clash_dtm.toarray(),
                      columns = vocab)

In [ ]:
df_dtm_positive = lexicon_df_dtm[lexicon_df_dtm.columns.intersection(opinion_lexicon.positive())]
df_dtm_positive.sum(axis = 1)
df_dtm_negative = lexicon_df_dtm[lexicon_df_dtm.columns.intersection(opinion_lexicon.negative())]
df_dtm_negative.sum(axis = 1)

,0
0,0
1,1
2,1
3,0
4,2
...,...
499995,0
499996,1
499997,0
499998,0


In [ ]:
positive_sent = df_dtm_positive.sum(axis = 1) - df_dtm_negative.sum(axis = 1) > 0

In [ ]:
sampledf['Prediction'] = np.where(positive_sent, 'positive', 'negative')

In [ ]:
np.mean(sampledf['review_category'] == sampledf['Prediction'])

np.float64(0.678116)

In [ ]:
# Bigram Setup
vectorizer = CountVectorizer(ngram_range = (2,2))
dtm_bg = vectorizer.fit_transform(smaller_sampledf['content'].values.astype('U'))
vocab_bg = vectorizer.get_feature_names_out().tolist()
df_dtm_bg = pd.DataFrame(data = dtm_bg.toarray(),
                      columns = vocab_bg)

In [ ]:
# most frequent bigrams

bigram_frequency = df_dtm_bg.sum().sort_values(ascending=False)
bigram_frequency.head(20)

In [ ]:
# Trigram Setup
vectorizer = CountVectorizer(ngram_range = (3,3))
dtm_tg = vectorizer.fit_transform(smaller_sampledf['content'].values.astype('U'))
vocab_tg = vectorizer.get_feature_names_out().tolist()
df_dtm_tg = pd.DataFrame(data = dtm_tg.toarray(),
                      columns = vocab_tg)

In [ ]:
# most frequent trigrams

trigram_frequency = df_dtm_tg.sum().sort_values(ascending=False)
trigram_frequency.head(20)

In [ ]:
df_positive = df[df["score"] > 2]
df_negative = df[df["score"] < 3]
print(df_positive["score"].describe())
print(df_negative["score"].describe())

count    1.937404e+06
mean     4.765609e+00
std      5.394465e-01
min      3.000000e+00
25%      5.000000e+00
50%      5.000000e+00
75%      5.000000e+00
max      5.000000e+00
Name: score, dtype: float64
count    369692.000000
mean          1.161388
std           0.367904
min           0.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           2.000000
Name: score, dtype: float64


In [ ]:
#positive trigrams
positive_sampledf = df_positive.sample(n = 100000, random_state = 123)
vectorizer = CountVectorizer(ngram_range = (3,3))
dtm_positive_tg = vectorizer.fit_transform(positive_sampledf['content'].values.astype('U'))
vocab_positive_tg = vectorizer.get_feature_names_out().tolist()
df_dtm_positive_tg = pd.DataFrame(data = dtm_positive_tg.toarray(),
                      columns = vocab_positive_tg)

In [ ]:
# most frequent positive review trigrams

positive_trigram_frequency = df_dtm_positive_tg.sum().sort_values(ascending=False)
positive_trigram_frequency.head(20)

In [ ]:
#negative trigrams
negative_sampledf = df_negative.sample(n = 100000, random_state = 123)
vectorizer = CountVectorizer(ngram_range = (3,3))
dtm_negative_tg = vectorizer.fit_transform(negative_sampledf['content'].values.astype('U'))
vocab_negative_tg = vectorizer.get_feature_names_out().tolist()
df_dtm_negative_tg = pd.DataFrame(data = dtm_negative_tg.toarray(),
                      columns = vocab_negative_tg)

In [ ]:
# most frequent negative review trigrams

negative_trigram_frequency = df_dtm_negative_tg.sum().sort_values(ascending=False)
negative_trigram_frequency.head(20)

,0
pay to win,9919
this game is,5604
the game is,2535
to win game,1679
play this game,1396
used to be,1340
worst game ever,1283
playing this game,1145
clash of clans,1124
in this game,977


In [ ]:
classify_data_df = pd.DataFrame(data = smaller_sampledf)
classify_data_df = classify_data_df[classify_data_df.columns[[1,6]]]

In [ ]:
# classifier for reviews
tokenizer = CountVectorizer(token_pattern="[^\W\d_]+", min_df=0.0, max_df = 1.0)
classifier_dtm = tokenizer.fit_transform(classify_data_df['content'].values.astype('U'))

classifier_df_dtm = pd.DataFrame(data = classifier_dtm.toarray(),
                      columns = tokenizer.get_feature_names_out().tolist())

<>:2: SyntaxWarning: invalid escape sequence '\W'
<>:2: SyntaxWarning: invalid escape sequence '\W'
/tmp/ipython-input-3666359050.py:2: SyntaxWarning: invalid escape sequence '\W'
  tokenizer = CountVectorizer(token_pattern="[^\W\d_]+", min_df=0.0, max_df = 1.0)


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(classify_data_df['content'], classify_data_df['review_category'], test_size = 0.2, random_state=123)
print(X_train)


1526701                                             It's fun
704538                                             Very good
470988                                           a good game
248845                                                  Good
745058                                             good game
                                 ...                        
1371353                        I love it when they add 2 v 2
1759412    It's awesome. You keep battling and keep winni...
2188655    SUPERCELL.......gr8 game ,an awesome idea ..re...
511959                                             Very nice
2262356    Play as much as you want, easy neat and huge v...
Name: content, Length: 80000, dtype: object


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn import metrics


text_clf = Pipeline([('vect', CountVectorizer()),
                     ('clf', LogisticRegression(random_state=123, solver='newton-cg')),
])

text_clf.fit(X_train.values.astype('U'), y_train.values.astype('U'))

predicted = text_clf.predict(X_test)

print(metrics.accuracy_score(y_test, predicted))


0.8999


In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, predicted))

              precision    recall  f1-score   support

    negative       0.77      0.54      0.64      3241
    positive       0.92      0.97      0.94     16759

    accuracy                           0.90     20000
   macro avg       0.84      0.75      0.79     20000
weighted avg       0.89      0.90      0.89     20000



In [ ]:
# Random Forest Model
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn import metrics

text_clf_rf = Pipeline([('vect', CountVectorizer()),
                        ('rf', RandomForestClassifier(random_state=123)),
])
text_clf_rf.fit(X_train.astype('U'), y_train.astype('U'))
predicted_rf = text_clf_rf.predict(X_test)
print(metrics.accuracy_score(y_test, predicted_rf))

0.89475


In [ ]:

text_clf_rf['rf'].feature_importances_

array([3.45772676e-06, 3.55633260e-05, 2.32486264e-07, ...,
       2.41048233e-08, 2.10089052e-07, 9.17487549e-08])